In [ ]:
!pip install polars pyarrow scikit-learn matplotlib seaborn

In [1]:
import polars as pl

# Carregar o arquivo leve de features prontas
df = pl.read_parquet("features/ml_features_1m_v2.parquet")
print(df.shape)
df.head()

(5587547, 16)


market_id,minute_bar,close_mid,mean_spread,close_spread,bar_volatility,total_volume,buy_volume,sell_volume,trade_count,order_flow_imbalance,target,return_1m,bid_depth,ask_depth,depth_imbalance
str,"datetime[μs, UTC]",f32,f32,f32,f32,f32,f32,f32,u32,f32,i8,f32,f64,f64,f64
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:00:00 UTC,0.285,0.45375,0.45,0.005,0.0,0.0,0.0,0,0.0,0,0.017857,287.79,3845.07,-0.860731
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:01:00 UTC,0.285,0.45,0.45,0.0,0.0,0.0,0.0,0,0.0,0,0.0,287.79,3845.07,-0.860731
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:02:00 UTC,0.29,0.443333,0.44,0.005,0.0,0.0,0.0,0,0.0,0,0.017544,287.79,3845.07,-0.860731
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:03:00 UTC,0.29,0.44,0.44,0.0,0.0,0.0,0.0,0,0.0,0,0.0,287.79,3845.07,-0.860731
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:04:00 UTC,0.29,0.44,0.44,0.0,0.0,0.0,0.0,0,0.0,0,0.0,366.8,8838.88,-0.92031


In [2]:
# Obter o esquema do DataFrame
schema = df.collect_schema()

for col, dtype in df.schema.items():
    print(f"{col}: {dtype}")

market_id: String
minute_bar: Datetime(time_unit='us', time_zone='UTC')
close_mid: Float32
mean_spread: Float32
close_spread: Float32
bar_volatility: Float32
total_volume: Float32
buy_volume: Float32
sell_volume: Float32
trade_count: UInt32
order_flow_imbalance: Float32
target: Int8
return_1m: Float32
bid_depth: Float64
ask_depth: Float64
depth_imbalance: Float64


**DESCRIÇÃO:**


```
market_id: Identificador único do mercado da Polymarket. Diferencia cada contrato de previsão (ex.: eleições, esportes, criptomoedas etc.). 
minute_bar: Timestamp correspondente ao início da barra temporal de 1 minuto utilizada na agregação das features.
close_mid: Preço médio de fechamento (mid-price) da barra. Calculado como a média entre o melhor preço de compra (best bid) e o melhor preço de venda (best ask).
mean_spread: Spread médio da barra de 1 minuto. Mede o custo médio implícito de negociação durante o período.
close_spread: Spread observado no fechamento da barra temporal.
bar_volatility: Volatilidade do preço dentro da barra de 1 minuto. Representa a intensidade das oscilações de preço durante o período.
total_volume: Volume total negociado na barra de 1 minuto.
buy_volume: Volume total de negociações iniciadas por compradores durante a barra.
sell_volume: Volume total de negociações iniciadas por vendedores durante a barra.
trade_count: Quantidade de negociações executadas durante a barra temporal.
order_flow_imbalance: Desequilíbrio do fluxo de ordens. Calculado como: (buy_volume - sell_volume) / total_volume. Valores positivos indicam predominância compradora; negativos indicam predominância vendedora.
return_1m: Retorno percentual do preço médio (mid-price) ao longo de 1 minuto.
bid_depth: Profundidade total do lado comprador do livro de ofertas (bid side), obtida a partir dos snapshots L2.
ask_depth: Profundidade total do lado vendedor do livro de ofertas (ask side), obtida a partir dos snapshots L2.
depth_imbalance: Desequilíbrio de profundidade do livro de ofertas. Calculado como: (bid_depth - ask_depth) / (bid_depth + ask_depth). Valores positivos indicam maior pressão compradora; negativos indicam maior pressão vendedora.
target: Variável alvo do problema de classificação. Assume valor 1 se o retorno futuro de 15 minutos for positivo e 0 caso contrário.
```


In [3]:
# Obter estatísticas descritivas do DataFrame
df.describe()

statistic,market_id,minute_bar,close_mid,mean_spread,close_spread,bar_volatility,total_volume,buy_volume,sell_volume,trade_count,order_flow_imbalance,target,return_1m,bid_depth,ask_depth,depth_imbalance
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""5587547""","""5587547""",5.587547e6,5.587547e6,5.587547e6,5.587547e6,5.587547e6,5.587547e6,5.587547e6,5.587547e6,5.587547e6,5.587547e6,5.587547e6,5.587547e6,5.587547e6,5.587547e6
"""null_count""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,"""2026-03-07 22:39:30.697299+00:…",0.264384,0.106947,0.105988,0.005248,16.70355,13.900503,2.803048,0.105569,11.097455,0.266501,0.358244,60529.263125,199718.031162,-0.337332
"""std""",null,null,0.280244,0.227499,0.227542,0.031491,862.310974,808.899719,270.595612,1.097412,843.505432,0.442129,18.620455,144176.553546,1.4187e6,0.557125
"""min""","""0x0007deb167d0bb816e2e847a1543…","""2026-03-06 00:00:00+00:00""",0.0005,0.000667,0.001,0.0,0.0,0.0,0.0,0.0,-344164.84375,0.0,-0.999,0.0,0.0,-1.0
"""25%""",null,"""2026-03-06 20:41:00+00:00""",0.0155,0.005,0.005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,486.78,3550.52,-0.843943
"""50%""",null,"""2026-03-07 17:22:00+00:00""",0.205,0.015,0.015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2845.32,17553.28,-0.424224
"""75%""",null,"""2026-03-08 17:40:00+00:00""",0.435,0.05,0.05,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,22510.5,90904.98,0.000912
"""max""","""0xfff145c6201796960cac69aea3f6…","""2026-03-11 23:59:00+00:00""",0.9995,1.0,1.0,0.9855,680889.0,680884.0,354123.78125,443.0,680879.0,1.0,998.999939,5.0261e6,2.1935e7,1.0


In [4]:
# Verificação da distribuição da variável alvo (target) no dataset.
# Objetivo: Avaliar o balanceamento entre as classes para direcionar a estratégia de validação e escolha de métricas.
# Resultado: Identificado desbalanceamento ~73% (Classe 0) vs ~27% (Classe 1).
# Implicação: Acurácia pura será uma métrica enganosa; usaremos ROC-AUC e F1-Score para avaliar os modelos.
print(df.group_by("target").len())

df.group_by("target").len().with_columns(
    (
        pl.col("len") /
        pl.col("len").sum()
    ).alias("proportion")
)

shape: (2, 2)
┌────────┬─────────┐
│ target ┆ len     │
│ ---    ┆ ---     │
│ i8     ┆ u32     │
╞════════╪═════════╡
│ 0      ┆ 4098458 │
│ 1      ┆ 1489089 │
└────────┴─────────┘


target,len,proportion
i8,u32,f64
0,4098458,0.733499
1,1489089,0.266501


In [5]:
# Verificar nulos
df.null_count()

market_id,minute_bar,close_mid,mean_spread,close_spread,bar_volatility,total_volume,buy_volume,sell_volume,trade_count,order_flow_imbalance,target,return_1m,bid_depth,ask_depth,depth_imbalance
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [6]:
# Verificar duplicatas
print(
    f"Duplicatas: {df.is_duplicated().sum()}"
)

Duplicatas: 0


In [7]:
# Verificar valores infinitos

numeric_cols = [
    c for c, t in df.schema.items()
    if t in (
        pl.Float32,
        pl.Float64
    )
]

for col in numeric_cols:
    print(
        col,
        df.filter(
            pl.col(col).is_infinite()
        ).height
    )

close_mid 0
mean_spread 0
close_spread 0
bar_volatility 0
total_volume 0
buy_volume 0
sell_volume 0
order_flow_imbalance 0
return_1m 0
bid_depth 0
ask_depth 0
depth_imbalance 0


In [8]:
# Verificar se o target é binário
df.select(
    pl.col("target").unique().sort()
)

target
i8
0
1


In [9]:
# Análise estatística da densidade do histórico de dados por mercado (linhas por market_id).
# Objetivo: Identificar a quantidade de observações temporais por ativo para filtrar mercados com histórico insuficiente.
# Resultado: Alta dispersão (média ~1186, mas mediana é 662). Há mercados com apenas 1 minuto de dados (min=1) e mercados altamente líquidos com até 8.221 minutos (max=8221).
# Implicação: Obrigatório aplicar um filtro de corte (ex: manter apenas mercados com len >= 1440 minutos / 1 dia completo) para garantir dados suficientes para o treinamento de modelos sequenciais e exibição no dashboard.
print(df.group_by("market_id").len().describe())
df.group_by("market_id").len().sort("len").head()
print(df.group_by("market_id").len().describe())

df.group_by("market_id").len().sort("len").head()

shape: (9, 3)
┌────────────┬─────────────────────────────────┬─────────────┐
│ statistic  ┆ market_id                       ┆ len         │
│ ---        ┆ ---                             ┆ ---         │
│ str        ┆ str                             ┆ f64         │
╞════════════╪═════════════════════════════════╪═════════════╡
│ count      ┆ 4710                            ┆ 4710.0      │
│ null_count ┆ 0                               ┆ 0.0         │
│ mean       ┆ null                            ┆ 1186.315711 │
│ std        ┆ null                            ┆ 1314.819778 │
│ min        ┆ 0x0007deb167d0bb816e2e847a1543… ┆ 1.0         │
│ 25%        ┆ null                            ┆ 296.0       │
│ 50%        ┆ null                            ┆ 662.0       │
│ 75%        ┆ null                            ┆ 1588.0      │
│ max        ┆ 0xfff145c6201796960cac69aea3f6… ┆ 8221.0      │
└────────────┴─────────────────────────────────┴─────────────┘
shape: (9, 3)
┌────────────┬─────────────

market_id,len
str,u32
"""0xaf617e83074e3536fa1b86b1190b…",1
"""0xec363de4b390604e844387471ab5…",3
"""0xb0ea878453eb9c7a835e6820fd35…",3
"""0x563ecbc4fcc34f60103e9c5b3354…",3
"""0xdfd03dee4fcf319bbda745eeb317…",4


A análise da distribuição de observações por mercado identificou 4.710 mercados distintos. Embora a maioria possua centenas de observações temporais, alguns mercados apresentam apenas poucas barras de negociação (entre 1 e 4 observações). Esses registros foram mantidos nesta etapa por não representarem erros de qualidade dos dados. A eventual exclusão de mercados com histórico insuficiente será avaliada em etapas posteriores de modelagem.

In [10]:
# Veificar datas
# Objetivo: Identificar as datas de início e fim do intervalo para planejar a divisão cronológica de treino e teste.
# Resultado: O subconjunto de dados cobre exatamente 6 dias completos, de 06/03/2026 a 11/03/2026.
# Implicação: Ideal para validação inicial (ex: usar os primeiros 4 dias para treino e os 2 últimos para teste).
df.select(
    pl.min("minute_bar").alias("inicio"),
    pl.max("minute_bar").alias("fim")
)

inicio,fim
"datetime[μs, UTC]","datetime[μs, UTC]"
2026-03-06 00:00:00 UTC,2026-03-11 23:59:00 UTC


In [11]:
# Contagem de mercados únicos presentes no subset de dados.
# Objetivo: Identificar a quantidade de ativos/contratos individuais disponíveis para mapear a cardinalidade dos dados.
# Resultado: Encontrados 4.710 mercados distintos ativos no período analisado.
# Implicação: Como cada mercado tem seu próprio comportamento, precisaremos agrupar por 'market_id' ao criar features temporais ou selecionar os mais líquidos para o dashboard.
df.select(
    pl.col("market_id").n_unique()
)

market_id
u32
4710
